# CDK20とCDK2/CDK3/CDK5/CDK6/CDK9のAlphaFold構造アラインメント


## Why

`cdk20_similar_targets.ipynb`のBLAST検索(`expect=1e-3`, `max_hits=200`)で見
つかったヒトパラログのうち、データが豊富な5つ -- `CDK2_HUMAN`(522構造/3454
活性)、`CDK9_HUMAN`(28構造/2247活性)、`CDK6_HUMAN`(22構造/1053活性)、
`CDK5_HUMAN`(46.0%同一性で最高、10構造/848活性)、`CDK3_HUMAN`(45.4%同一性、
2構造/59活性) -- のAlphaFold予測構造をダウンロードし、CDK20自身のAlphaFold
モデルを基準に`chem.protein.align`で立体的に重ね合わせる。CDK20には実験構造
が1つも無いため、他のノートブックのように複数のRCSB構造を1つのAlphaFold参
照に揃える構図と対称的に、ここでは**複数の異なる(しかし相同な)蛋白質**を
CDK20という1つの参照に揃える。`chem.protein.align`は配列アラインメントで
マッチした(ギャップの無い)CA原子ペアだけをKabsch重ね合わせに使うため、対象
が「同一蛋白質の異なる構造」である必要はなく、保存されたキナーゼドメインの
コア部分さえ十分にマッチすれば異なるパラログ同士でも機能する。

残基レベルの違い(特にCDK20のポケット周辺に限定した比較)は、この重ね合わせ
が済んでからの次のステップ。

## Downloading each target's AlphaFold model

`Q8IZL9`(CDK20)含め、ここで扱う6ターゲットは全てUniProt上に複数のスプライ
スアイソフォームを持ち、AlphaFold DBはそれぞれを別エントリとして予測してい
る(例: `CDK9_HUMAN`は`AF-P50750-F1`(372残基、正準)と`AF-P50750-2-F1`
(489残基、アイソフォーム2)の2エントリ)。他のCDK20系ノートブックと同じ理
由・同じ正規表現で、正準エントリ(`AF-<accession>-F<fragment>.pdb`、`AF`と
フラグメント番号の間にアイソフォーム番号が挟まらないもの)を明示的に選ぶ。

ダウンロード先は`cdk_paralogs_af_data/`(このノートブック専用 -- 他のCDK20
ノートブックの`cdk20_af_data/`や、thrombinノートブックの`af_data/`とは別デ
ィレクトリにして、ノートブック間でのファイル衝突を避ける)。

In [ ]:
import os
import re

from chem import alphafold, ids

TARGETS = ["CDK20_HUMAN", "CDK2_HUMAN", "CDK9_HUMAN", "CDK6_HUMAN", "CDK5_HUMAN", "CDK3_HUMAN"]
OUTDIR = "cdk_paralogs_af_data"

accessions = {name: ids.resolve_uniprot_accession(name) for name in TARGETS}
print(accessions)

for name in TARGETS:
    alphafold.download_structures(name, outdir=OUTDIR, filetype="pdb")

In [ ]:
_canonical_re = re.compile(r"^AF-([^-]+)-F\d+\.pdb$")


def canonical_file(accession, outdir):
    matches = [
        f for f in os.listdir(outdir) if (m := _canonical_re.match(f)) and m.group(1) == accession
    ]
    if len(matches) != 1:
        raise ValueError(f"expected exactly one canonical AlphaFold entry for {accession}, found {matches}")
    return matches[0]


canonical_files = {name: canonical_file(accessions[name], OUTDIR) for name in TARGETS}
canonical_files

## Structural alignment via chem.protein.align

`chem.protein.align`は、参照(`reference`)の主鎖CA原子と各対象構造のCA原子
を配列アラインメントでマッチさせ、マッチした(ギャップの無い)ペアだけを使っ
てKabsch法で剛体重ね合わせを行う。参照をCDK20自身のAlphaFoldモデルに固定
し、5つのパラログをそれぞれ重ね合わせる。返り値の`identity`は、マッチした
位置のうち同一残基だった割合 -- BLAST検索の`identity_pct`と近い値になるは
ずだが、計算方法が異なる(BLASTはBLOSUM62スコアに基づく局所アラインメン
ト、こちらはCA座標が実際にペアリングされた位置での単純な一致率)ので、厳密
には一致しない。

In [ ]:
from chem import protein
import pandas as pd

reference_path = os.path.join(OUTDIR, canonical_files["CDK20_HUMAN"])
paralog_names = [t for t in TARGETS if t != "CDK20_HUMAN"]
paralog_paths = [os.path.join(OUTDIR, canonical_files[name]) for name in paralog_names]

align_results = protein.align(paralog_paths, reference=reference_path, outdir="cdk_paralogs_aligned")

# align() includes an entry for the reference itself (rmsd=0.0, identity=1.0),
# so path_to_name must cover it too, not just the paralog paths.
path_to_name = dict(zip([reference_path] + paralog_paths, ["CDK20_HUMAN"] + paralog_names))
align_df = pd.DataFrame(
    [
        {"target": path_to_name[p], "rmsd": r["rmsd"], "identity": r["identity"]}
        for p, r in align_results.items()
    ]
).sort_values("rmsd")
display(align_df.style.hide(axis="index").format({"rmsd": "{:.3f}", "identity": "{:.1%}"}))

### Overlaying the aligned structures

Quick visual sanity check: every paralog's cartoon backbone, superposed onto
CDK20's (highlighted in a distinct color), all in the same frame written by
`align()` into `cdk_paralogs_aligned/`.

In [ ]:
import py3Dmol
from IPython.display import HTML, display

_colors = {
    "CDK20_HUMAN": "black",
    "CDK2_HUMAN": "orange",
    "CDK9_HUMAN": "cyan",
    "CDK6_HUMAN": "magenta",
    "CDK5_HUMAN": "green",
    "CDK3_HUMAN": "purple",
}

view = py3Dmol.view(width=650, height=500)
legend_html = ""
for name in TARGETS:
    path = os.path.join("cdk_paralogs_aligned", canonical_files[name])
    with open(path) as f:
        pdb_text = f.read()
    view.addModel(pdb_text, "pdb")
    color = _colors[name]
    view.setStyle({"model": -1}, {"cartoon": {"color": color}})
    legend_html += (
        f'<div><span style="display:inline-block;width:12px;height:12px;background:{color};'
        f'border:1px solid #888;margin-right:6px;"></span>{name}</div>'
    )
view.zoomTo()

display(HTML(f'<div style="display:flex; gap:16px;">'
             f'<div id="cdk-paralog-overlay" style="width:650px; height:500px; border:1px solid #ccc;"></div>'
             f'<div>{legend_html}</div></div>'))
view.insert("cdk-paralog-overlay")